In [9]:
import pandas as pd
import numpy as np

features = pd.read_csv('../data/processed/features.csv', parse_dates=['date'])

print(f"Shape: {features.shape}")
print(f"\nDate range: {features['date'].min().date()} → {features['date'].max().date()}")
print(f"\nDtypes:\n{features.dtypes}")
print(f"\nMissing values:\n{features.isnull().sum()}")
print(f"\nSample:\n{features.head(3)}")

Shape: (16610, 15)

Date range: 1916-07-02 → 2026-06-18

Dtypes:
date                       datetime64[us]
home_team                             str
away_team                             str
neutral                              bool
tournament                            str
home_win_rate                     float64
home_avg_goals_scored             float64
home_avg_goals_conceded           float64
away_win_rate                     float64
away_avg_goals_scored             float64
away_avg_goals_conceded           float64
elo_diff                            int64
home_elo                            int64
away_elo                            int64
result                                str
dtype: object

Missing values:
date                       0
home_team                  0
away_team                  0
neutral                    0
tournament                 0
home_win_rate              0
home_avg_goals_scored      0
home_avg_goals_conceded    0
away_win_rate              0
away_avg_goal

In [10]:
FEATURE_COLS = [
    'home_win_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded',
    'away_win_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded',
    'elo_diff', 'neutral', 'tournament',
]
TARGET_COL = 'result'

WC_2022_START = pd.Timestamp('2022-11-20')
WC_2026_START = pd.Timestamp('2026-06-11')

train = features[features['date'] < WC_2022_START].copy()
val   = features[
    (features['date'] >= WC_2022_START) &
    (features['date'] <  WC_2026_START) &
    (features['tournament'] == 'FIFA World Cup')
].copy()
test  = features[features['date'] >= WC_2026_START].copy()

print(f"Train:      {len(train):>5,} matches  ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Validation: {len(val):>5,} matches  ({val['date'].min().date()} → {val['date'].max().date()})")
print(f"Test:       {len(test):>5,} matches  ({test['date'].min().date()} → {test['date'].max().date()})")

print(f"\nVal result distribution:\n{val[TARGET_COL].value_counts()}")
print(f"\nTest result distribution:\n{test[TARGET_COL].value_counts()}")

Train:      14,639 matches  (1916-07-02 → 2022-09-27)
Validation:    64 matches  (2022-11-20 → 2022-12-18)
Test:          28 matches  (2026-06-11 → 2026-06-18)

Val result distribution:
result
home_win    28
away_win    21
draw        15
Name: count, dtype: int64

Test result distribution:
result
home_win    15
draw        10
away_win     3
Name: count, dtype: int64


In [16]:
from sklearn.preprocessing import LabelEncoder

# Target encoding — explicit map to fixed integers
RESULT_MAP     = {'home_win': 0, 'draw': 1, 'away_win': 2}
RESULT_MAP_INV = {0: 'home_win', 1: 'draw', 2: 'away_win'}

# Tournament encoding — fit on full dataset so all 10 values are known
le_tournament = LabelEncoder()
le_tournament.fit(features['tournament'])

def encode_X(df):
    X = df[FEATURE_COLS].copy()
    X['neutral']    = X['neutral'].astype(int)
    X['tournament'] = le_tournament.transform(X['tournament'])
    return X

def encode_y(df):
    return df[TARGET_COL].map(RESULT_MAP)

X_train = encode_X(train)
y_train = encode_y(train)
X_val   = encode_X(val)
y_val   = encode_y(val)
X_test  = encode_X(test)
y_test  = encode_y(test)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print(f"\ny_train distribution:\n{y_train.value_counts().sort_index().rename(RESULT_MAP_INV)}")

print("\nTournament encoding:")
for code, name in enumerate(le_tournament.classes_):
    print(f"  {code}: {name}")

X_train: (14639, 9)
X_val:   (64, 9)
X_test:  (28, 9)

y_train distribution:
result
home_win    7256
draw        3142
away_win    4241
Name: count, dtype: int64

Tournament encoding:
  0: AFC Asian Cup
  1: African Cup of Nations
  2: CONCACAF Nations League
  3: Copa América
  4: FIFA World Cup
  5: FIFA World Cup qualification
  6: Gold Cup
  7: UEFA Euro
  8: UEFA Euro qualification
  9: UEFA Nations League


In [17]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    early_stopping_rounds=30,
    random_state=42,
    verbosity=0,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

print(f"\nBest iteration:    {model.best_iteration}")
print(f"Best val log loss: {model.best_score:.4f}")

[0]	validation_0-mlogloss:1.07169
[50]	validation_0-mlogloss:1.02167
[100]	validation_0-mlogloss:1.02156
[115]	validation_0-mlogloss:1.02187

Best iteration:    85
Best val log loss: 1.0195


In [18]:
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

LABELS = ['home_win', 'draw', 'away_win']

def evaluate(name, X, y_true):
    y_pred      = model.predict(X)
    y_proba     = model.predict_proba(X)
    acc         = accuracy_score(y_true, y_pred)
    ll          = log_loss(y_true, y_proba, labels=[0, 1, 2])
    baseline    = max(y_true.value_counts(normalize=True))

    print(f"=== {name} ===")
    print(f"Accuracy:          {acc:.3f}  (baseline always-majority: {baseline:.3f})")
    print(f"Log loss:          {ll:.4f}  (random 3-class: {1.0986:.4f})")
    print()
    print(classification_report(
        y_true, y_pred,
        target_names=LABELS,
        zero_division=0,
    ))
    print("Confusion matrix (rows=actual, cols=predicted):")
    print(pd.DataFrame(
        confusion_matrix(y_true, y_pred),
        index=LABELS, columns=LABELS,
    ))
    print()

evaluate("Validation — WC 2022", X_val, y_val)
evaluate("Test       — WC 2026", X_test, y_test)

=== Validation — WC 2022 ===
Accuracy:          0.484  (baseline always-majority: 0.438)
Log loss:          1.0195  (random 3-class: 1.0986)

              precision    recall  f1-score   support

    home_win       0.49      0.79      0.60        28
        draw       0.00      0.00      0.00        15
    away_win       0.47      0.43      0.45        21

    accuracy                           0.48        64
   macro avg       0.32      0.40      0.35        64
weighted avg       0.37      0.48      0.41        64

Confusion matrix (rows=actual, cols=predicted):
          home_win  draw  away_win
home_win        22     0         6
draw            11     0         4
away_win        12     0         9

=== Test       — WC 2026 ===
Accuracy:          0.536  (baseline always-majority: 0.536)
Log loss:          0.9940  (random 3-class: 1.0986)

              precision    recall  f1-score   support

    home_win       0.60      0.80      0.69        15
        draw       0.00      0.00    

In [19]:
importance = pd.Series(
    model.feature_importances_,
    index=FEATURE_COLS,
).sort_values(ascending=False)

print("Feature importance:")
print(importance.round(4).to_string())

Feature importance:
elo_diff                   0.2028
home_avg_goals_conceded    0.1612
away_avg_goals_conceded    0.1313
home_win_rate              0.1274
away_win_rate              0.1064
away_avg_goals_scored      0.0862
neutral                    0.0787
home_avg_goals_scored      0.0698
tournament                 0.0362


In [20]:
import pickle
from pathlib import Path

Path('../models').mkdir(exist_ok=True)

artifact = {
    'model':          model,
    'le_tournament':  le_tournament,
    'feature_cols':   FEATURE_COLS,
    'result_map':     RESULT_MAP,
    'result_map_inv': RESULT_MAP_INV,
}

with open('../models/xgb_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print("Saved: models/xgb_model.pkl")
print(f"Contents: {list(artifact.keys())}")

Saved: models/xgb_model.pkl
Contents: ['model', 'le_tournament', 'feature_cols', 'result_map', 'result_map_inv']


In [22]:
import sys
sys.path.insert(0, '..')

from src.predict import predict_match

fixtures = pd.read_csv('../data/processed/wc_2026_fixtures_enriched.csv')
group_stage = fixtures[fixtures['is_placeholder_match'] == False].copy()

rows = []
for _, match in group_stage.iterrows():
    pred = predict_match(match['team1'], match['team2'], neutral=True)
    rows.append({
        'group':          match['group'],
        'date':           match['date'],
        'team1':          match['team1'],
        'team2':          match['team2'],
        'team1_win_prob': pred['home_win'],
        'draw_prob':      pred['draw'],
        'team2_win_prob': pred['away_win'],
        'favorite':       match['team1'] if pred['favorite'] == 'home_win'
                          else (match['team2'] if pred['favorite'] == 'away_win' else 'draw'),
    })

predictions = pd.DataFrame(rows).sort_values(['group', 'date']).reset_index(drop=True)
predictions.to_csv('../data/processed/predictions_2026.csv', index=False)

print(f"Saved: predictions_2026.csv  ({len(predictions)} matches)")
print()
print(predictions[['group', 'date', 'team1', 'team2', 'team1_win_prob', 'draw_prob', 'team2_win_prob', 'favorite']].to_string(index=False))

Saved: predictions_2026.csv  (72 matches)

group       date                  team1                  team2  team1_win_prob  draw_prob  team2_win_prob               favorite
    A 2026-06-11                 Mexico           South Africa           0.600      0.238           0.162                 Mexico
    A 2026-06-11            South Korea                Czechia           0.530      0.242           0.228            South Korea
    A 2026-06-18            South Korea                 Mexico           0.341      0.292           0.367                 Mexico
    A 2026-06-18                Czechia           South Africa           0.504      0.244           0.252                Czechia
    A 2026-06-24                Czechia                 Mexico           0.275      0.252           0.473                 Mexico
    A 2026-06-24           South Africa            South Korea           0.252      0.248           0.501            South Korea
    B 2026-06-12                 Canada Bosnia and Her